# Viscan Dental OPG AI Assistant — GPU Training & Evaluation
**AI Engineer Take-Home Assessment (Round 2)**

> **Clinical Notice:** AI-assisted findings for dentist review only. Not a diagnosis.

## Task
Detect four dental pathologies in 2D panoramic radiographs (OPGs):
1. Impacted Tooth  2. Caries  3. Periapical Lesion  4. Deep Caries

## Dataset
- **Source:** DENTEX (MICCAI 2023) — `huggingface.co/datasets/ibrahimhamamci/DENTEX`
- **License:** CC BY-NC-SA 4.0
- **Split:** 635 train / 70 val (checkpoint selection) / 50 held-out test (not used for threshold or checkpoint selection)
- **Preprocessing:** COCO-format conversion, bbox clipping to image raster
- **Augmentation:** Random horizontal flip (p=0.5) + color jitter (brightness/contrast ±0.15)
- **Class imbalance:** Caries:Periapical ≈ 13.9:1 in train (no resampling applied; noted as limitation)

## Model Selection Rationale

| Criterion | Faster R-CNN ResNet-50 FPN V2 **(Selected)** | YOLOv11n (Alternative) |
|---|---|---|
| Architecture | Two-stage detector with FPN, ResNet-50 backbone | Single-stage anchor-free |
| Pretrained Weights | torchvision COCO_V1 | Ultralytics COCO |
| Code License | **BSD-3-Clause** — commercial-safe | **AGPL-3.0** — copyleft, requires Enterprise licence |
| Parameters | 43.3M (stronger feature extraction for fine details) | ~2.6M |
| Deployment | Pure PyTorch, zero C++ deps | Requires Ultralytics ecosystem |

**License analysis:**
- **Code & architecture:** torchvision BSD-3-Clause — unrestricted commercial use.
- **Fine-tuned weights:** Trained on DENTEX (CC BY-NC-SA 4.0). This licence prohibits
  commercial use of derivative works. For commercial deployment, the model must be
  retrained on a permissively licensed dataset, or explicit permission obtained from
  the DENTEX consortium.
- **YOLOv11n:** AGPL-3.0 adds a code-level copyleft constraint on top of any data licence.

ResNet-50 FPN V2 chosen over MobileNetV3 for significantly stronger feature extraction on
high-resolution medical images where small lesion detection is critical.

Licence sources: [torchvision LICENSE](https://github.com/pytorch/vision/blob/main/LICENSE),
[ultralytics LICENSE](https://github.com/ultralytics/ultralytics/blob/main/LICENSE).

## Training Setup
- **Backbone:** ResNet-50 FPN V2 (COCO_V1 pretrained) with fine-tuned head
- **Augmentation:** Horizontal flip + brightness/contrast jitter
- **Optimizer:** SGD (lr=0.005, momentum=0.9, wd=5e-4) + cosine annealing LR
- **Mixed Precision:** AMP (FP16) for memory efficiency on T4 GPU
- **Epochs:** 12 (baseline & variant)

## Evaluation Protocol
- **Metrics:** COCO mAP@50, mAP@50:95, per-class AP, confusion matrix, P/R/F1
- **Threshold selection:** Macro-F1 sweep on val set → applied once to held-out test
- **Improvement experiment:** Higher input resolution (960×1600) vs baseline (800×1333)
- **Latency:** 30-run CPU benchmark on full-resolution OPG image

## Limitations & Future Work
- Periapical lesion has only 9 test samples — AP is statistically unreliable
- No class-weighted loss despite 13:1 imbalance (caries vs periapical)
- ONNX export + INT8 quantisation would reduce CPU latency significantly
- Active learning on false positive hotspots (cervical burnout artifacts) is recommended
- Multi-scale testing and test-time augmentation were not applied


In [ ]:
# 1. Install & Verify Environment
!pip install -q pydicom pycocotools pyyaml

import os, sys, json, time, random, gc
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image, ImageDraw
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import yaml

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
print(f"CPU Threads:     {torch.get_num_threads()}")


In [ ]:
# 2. Reproducibility: Seeds, Classes, Config YAML
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
g_seed = torch.Generator()
g_seed.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

CLASS_NAMES = {0: "background", 1: "impacted_tooth", 2: "caries", 3: "periapical_lesion", 4: "deep_caries"}
NUM_CLASSES = 5
EPOCHS = 12
LR = 0.005

config = {
    "seed": SEED,
    "data": {"dataset": "DENTEX (MICCAI 2023)", "source": "huggingface.co/datasets/ibrahimhamamci/DENTEX", "licence": "CC BY-NC-SA 4.0", "num_classes": NUM_CLASSES},
    "training": {"epochs": EPOCHS, "batch_size": 2, "lr": LR, "momentum": 0.9, "weight_decay": 0.0005, "lr_scheduler": "CosineAnnealingLR", "augmentation": "HFlip(0.5)+ColorJitter(0.15)", "amp": True},
    "model": {"name": "fasterrcnn_resnet50_fpn_v2", "pretrained": "COCO_V1", "licence": "BSD-3-Clause", "min_size": 800, "max_size": 1333, "box_score_thresh": 0.001},
}
os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print("Saved /kaggle/working/config.yaml")
print(yaml.dump(config, default_flow_style=False))


In [ ]:
# 3. DENTEX Dataset Download & Extraction
os.makedirs("/kaggle/working/data/dentex/annotations", exist_ok=True)
os.makedirs("/kaggle/temp", exist_ok=True)
BASE_DIR = Path("/kaggle/working/data/dentex")
ANN_DIR = BASE_DIR / "annotations"

print("Downloading DENTEX dataset files...")
# Pin to specific HF revision for reproducibility (replace "main" with commit hash before production)
DENTEX_BASE = "https://huggingface.co/datasets/ibrahimhamamci/DENTEX/resolve/main/DENTEX"
!wget -q --show-progress -O /kaggle/working/data/dentex/val_raw.json {DENTEX_BASE}/validation_triple.json
!wget -q --show-progress -O /kaggle/temp/validation_data.zip {DENTEX_BASE}/validation_data.zip
!wget -q --show-progress -O /kaggle/temp/training_data.zip {DENTEX_BASE}/training_data.zip
print("Extracting validation data...")
!cd /kaggle/working/data/dentex && unzip -q -o /kaggle/temp/validation_data.zip
print("Extracting training diagnosis data...")
!cd /kaggle/working/data/dentex && unzip -q -o /kaggle/temp/training_data.zip "training_data/quadrant-enumeration-disease/*"
!rm -f /kaggle/temp/training_data.zip /kaggle/temp/validation_data.zip
print("Verifying extracted files...")
!find /kaggle/working/data/dentex -name "*.png" ! -path "*checkpoint*" | wc -l
print("Extraction complete!")


In [ ]:
# 4. COCO Conversion, 3-Way Split, Leakage Check, Data Card
CATEGORY_MAPPING = {0: {"id": 1, "name": "impacted_tooth"}, 1: {"id": 2, "name": "caries"}, 2: {"id": 3, "name": "periapical_lesion"}, 3: {"id": 4, "name": "deep_caries"}}
cats = [{"id": v["id"], "name": v["name"]} for v in CATEGORY_MAPPING.values()]

# Resolve JSON paths
train_cands = list(BASE_DIR.rglob("*disease.json"))
train_json_path = train_cands[0] if train_cands else BASE_DIR / "training_data" / "quadrant-enumeration-disease" / "train_quadrant_enumeration_disease.json"
test_json_path = BASE_DIR / "val_raw.json"
if not test_json_path.exists():
    tc = list(BASE_DIR.rglob("*triple.json"))
    if tc:
        test_json_path = tc[0]
print(f"Train source JSON: {train_json_path}")
print(f"Test source JSON:  {test_json_path}")

# Convert raw DENTEX to COCO format
def raw_to_coco(path):
    raw = json.load(open(str(path), "r", encoding="utf-8"))
    imgs = {i["id"]: i for i in raw.get("images", [])}
    anns, aid = [], 1
    for a in raw.get("annotations", []):
        c3 = a.get("category_id_3", a.get("category_id"))
        if c3 not in CATEGORY_MAPPING:
            continue
        bb = a.get("bbox", [])
        if len(bb) != 4 or bb[2] <= 0 or bb[3] <= 0:
            continue
        anns.append({"id": aid, "image_id": a["image_id"], "category_id": CATEGORY_MAPPING[c3]["id"], "bbox": [float(v) for v in bb], "area": float(bb[2]*bb[3]), "iscrowd": 0})
        aid += 1
    return {"images": list(imgs.values()), "annotations": anns, "categories": cats}

full_train = raw_to_coco(train_json_path)
test_coco = raw_to_coco(test_json_path)
print(f"Full train pool: {len(full_train['images'])} images, {len(full_train['annotations'])} annotations")
print(f"Test (held-out): {len(test_coco['images'])} images, {len(test_coco['annotations'])} annotations")

# --- 3-Way Split: carve 70 val from 705, keep official 50 as TEST ---
all_img_ids = sorted([i["id"] for i in full_train["images"]])
rng = random.Random(SEED)
rng.shuffle(all_img_ids)
val_ids_set = set(all_img_ids[:70])
train_ids_set = set(all_img_ids[70:])

train_coco = {"images": [i for i in full_train["images"] if i["id"] in train_ids_set], "annotations": [a for a in full_train["annotations"] if a["image_id"] in train_ids_set], "categories": cats}
val_coco = {"images": [i for i in full_train["images"] if i["id"] in val_ids_set], "annotations": [a for a in full_train["annotations"] if a["image_id"] in val_ids_set], "categories": cats}

json.dump(train_coco, open(str(ANN_DIR / "instances_train.json"), "w", encoding="utf-8"))
json.dump(val_coco, open(str(ANN_DIR / "instances_val.json"), "w", encoding="utf-8"))
json.dump(test_coco, open(str(ANN_DIR / "instances_test.json"), "w", encoding="utf-8"))

print(f"\n=== 3-Way Split (seed={SEED}) ===")
print(f"Train: {len(train_coco['images'])} images, {len(train_coco['annotations'])} annotations")
print(f"Val:   {len(val_coco['images'])} images, {len(val_coco['annotations'])} annotations  (checkpoint selection only)")
print(f"Test:  {len(test_coco['images'])} images, {len(test_coco['annotations'])} annotations  (held-out; not used for threshold or checkpoint selection)")

# --- Leakage Check ---
train_fn = {os.path.basename(i.get("file_name","")) for i in train_coco["images"]}
val_fn = {os.path.basename(i.get("file_name","")) for i in val_coco["images"]}
test_fn = {os.path.basename(i.get("file_name","")) for i in test_coco["images"]}
print(f"\n=== Leakage Check (filename intersection) ===")
print(f"Train & Val:  {len(train_fn & val_fn)} overlap  {'LEAK' if train_fn & val_fn else 'clean'}")
print(f"Train & Test: {len(train_fn & test_fn)} overlap  {'LEAK' if train_fn & test_fn else 'clean'}")
print(f"NOTE: This check is filename-based only. DENTEX provides no patient IDs,")
print(f"so patient-level deduplication is not possible with available metadata.")
print(f"Val & Test:   {len(val_fn & test_fn)} overlap  {'LEAK' if val_fn & test_fn else 'clean'}")

# --- Per-Class Data Card ---
print(f"\n=== Data Card: Per-Class Annotation Counts ===")
print(f"{'Class':<20} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 48)
for cid in range(1, 5):
    cname = CLASS_NAMES[cid]
    tr = sum(1 for a in train_coco["annotations"] if a["category_id"] == cid)
    vl = sum(1 for a in val_coco["annotations"] if a["category_id"] == cid)
    te = sum(1 for a in test_coco["annotations"] if a["category_id"] == cid)
    print(f"  {cname:<18} {tr:>8} {vl:>8} {te:>8}")
tr_t = len(train_coco["annotations"])
vl_t = len(val_coco["annotations"])
te_t = len(test_coco["annotations"])
print(f"  {'TOTAL':<18} {tr_t:>8} {vl_t:>8} {te_t:>8}")

# --- Image Dimensions ---
print(f"\n=== Image Dimensions ===")
ws = [i.get("width", 0) for i in full_train["images"] if i.get("width")]
hs = [i.get("height", 0) for i in full_train["images"] if i.get("height")]
if ws and hs:
    print(f"  Width:  min={min(ws)}, max={max(ws)}, mean={np.mean(ws):.0f}")
    print(f"  Height: min={min(hs)}, max={max(hs)}, mean={np.mean(hs):.0f}")
    short = [min(w,h) for w,h in zip(ws,hs)]
    long_ = [max(w,h) for w,h in zip(ws,hs)]
    print(f"  Short side: min={min(short)}, max={max(short)}, mean={np.mean(short):.0f}")
    print(f"  Long side:  min={min(long_)}, max={max(long_)}, mean={np.mean(long_):.0f}")
    print(f"  With min_size=800, max_size=1333:")
    scales = [min(800/s, 1333/l) for s,l in zip(short, long_)]
    print(f"    Scale factor: min={min(scales):.3f}, max={max(scales):.3f}, mean={np.mean(scales):.3f}")
else:
    sample = full_train["images"][0]
    fname = os.path.basename(sample.get("file_name",""))
    pp = list(BASE_DIR.rglob(fname))
    if pp:
        w, h = Image.open(pp[0]).size
        print(f"  Sample ({fname}): {w} x {h}")
    else:
        print("  No dimension info available")


In [ ]:
# 5. Dataset, Model Factory, DataLoader Seeding
class DentexDataset(Dataset):
    def __init__(self, img_dir, ann_path, is_train=False):
        self.img_dir = Path(img_dir)
        self.coco = COCO(str(ann_path))
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.is_train = is_train
        self.color_jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        fname = os.path.basename(info["file_name"])
        p = list(self.img_dir.rglob(fname))
        if not p:
            p = list(BASE_DIR.rglob(fname))
        if not p:
            raise FileNotFoundError(f"Image {fname} not found")
        img = Image.open(p[0]).convert("RGB")
        w, h = img.size
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        boxes, labels = [], []
        for a in anns:
            x, y, bw, bh = a["bbox"]
            boxes.append([max(0, x), max(0, y), min(w, x + bw), min(h, y + bh)])
            labels.append(a["category_id"])
        if not boxes:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        # --- Training augmentation ---
        if self.is_train:
            # Random horizontal flip (p=0.5)
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if boxes.shape[0] > 0:
                    xmin_new = w - boxes[:, 2]
                    xmax_new = w - boxes[:, 0]
                    boxes[:, 0] = xmin_new
                    boxes[:, 2] = xmax_new
            # Subtle color jitter (brightness + contrast only, safe for radiographs)
            img = self.color_jitter(img)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([img_id])}
        return transforms.functional.to_tensor(img), target

def get_model(min_size=800, max_size=1333, score_thresh=0.001):
    m = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1,
        min_size=min_size, max_size=max_size,
        box_score_thresh=score_thresh, box_detections_per_img=300,
        box_nms_thresh=0.5, rpn_score_thresh=0.0,
        rpn_pre_nms_top_n_test=1000, rpn_post_nms_top_n_test=1000, rpn_nms_thresh=0.7)
    inf = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(inf, NUM_CLASSES)
    return m

def worker_init_fn(worker_id):
    seed = SEED + worker_id
    np.random.seed(seed)
    random.seed(seed)

def collate_fn(batch):
    return tuple(zip(*batch))

print("Dataset and model factory defined.")
print("Augmentation: HorizontalFlip(p=0.5) + ColorJitter(brightness=0.15, contrast=0.15)")
print("Backbone: ResNet-50 FPN V2 (BSD-3-Clause)")


In [ ]:
# 6. Baseline Training (635 train / 70 val for checkpoint selection)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

train_dataset = DentexDataset(BASE_DIR, ANN_DIR / "instances_train.json", is_train=True)
val_dataset = DentexDataset(BASE_DIR, ANN_DIR / "instances_val.json", is_train=False)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn, generator=g_seed, worker_init_fn=worker_init_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=0)

model = get_model(min_size=800, max_size=1333).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total_params:,} total, {trainable:,} trainable")

optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

train_losses, val_maps = [], []
best_map = 0.0

def run_val_eval(mdl, loader, ds):
    mdl.eval()
    results = []
    with torch.no_grad():
        for imgs, tgts in loader:
            imgs = [i.to(device) for i in imgs]
            outs = mdl(imgs)
            for tgt, out in zip(tgts, outs):
                for b, s, l in zip(out["boxes"].cpu().numpy(), out["scores"].cpu().numpy(), out["labels"].cpu().numpy()):
                    results.append({"image_id": int(tgt["image_id"].item()), "category_id": int(l), "bbox": [float(b[0]), float(b[1]), float(b[2]-b[0]), float(b[3]-b[1])], "score": float(s)})
    if not results:
        return 0.0, None
    coco_dt = ds.coco.loadRes(results)
    ev = COCOeval(ds.coco, coco_dt, "bbox")
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
    return float(ev.stats[1]), ev.stats

print(f"Training: {EPOCHS} epochs, CosineAnnealingLR, AMP={device.type=='cuda'}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    for imgs, tgts in train_loader:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            ld = model(imgs, tgts)
            loss = sum(v for v in ld.values())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item()
    lr_scheduler.step()
    mean_loss = loss_sum / len(train_loader)
    train_losses.append(mean_loss)

    cur_map, stats = run_val_eval(model, val_loader, val_dataset)
    val_maps.append(cur_map)

    ar100 = float(stats[8]) if stats is not None else 0.0
    map5095 = float(stats[0]) if stats is not None else 0.0
    map75 = float(stats[2]) if stats is not None else 0.0
    cur_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch}/{EPOCHS}] Loss: {mean_loss:.4f} | Val mAP@50: {cur_map:.3f} | mAP@50:95: {map5095:.3f} | mAP@75: {map75:.3f} | AR@100: {ar100:.3f} | lr: {cur_lr:.6f}")

    if cur_map >= best_map:
        best_map = cur_map
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print(f"  -> Saved best checkpoint (mAP@50={best_map:.3f})")

torch.save(model.state_dict(), "/kaggle/working/last_model.pth")
import shutil
shutil.copy("/kaggle/working/config.yaml", "/kaggle/working/config_baseline.yaml")
print(f"\nTraining complete. Best val mAP@50: {best_map:.3f}")


In [ ]:
# 7. Training Curves & Checkpoint Stats
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, train_losses, "#2196F3", marker="o", linewidth=2, markersize=6)
ax1.fill_between(epochs_range, train_losses, alpha=0.15, color="#2196F3")
ax1.set_title("Training Loss vs Epoch", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)
ax1.set_xticks(list(epochs_range))

ax2.plot(epochs_range, val_maps, "#4CAF50", marker="s", linewidth=2, markersize=6)
ax2.fill_between(epochs_range, val_maps, alpha=0.15, color="#4CAF50")
ax2.axhline(y=max(val_maps), color="#FF5722", linestyle="--", alpha=0.7, label=f"Best: {max(val_maps):.3f}")
ax2.set_title("Val mAP@50 vs Epoch (70-image val set)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("mAP@50")
ax2.grid(True, alpha=0.3)
ax2.set_xticks(list(epochs_range))
ax2.legend()

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Checkpoint file sizes
for f in ["best_model.pth", "last_model.pth", "config.yaml"]:
    fp = Path("/kaggle/working") / f
    if fp.exists():
        sz = os.path.getsize(fp)
        print(f"  {f}: {sz:,} bytes ({sz/1e6:.1f} MB)")


In [ ]:
# 8. Held-Out Test Evaluation (50 images, evaluated ONCE)
print("=== Loading best checkpoint for test evaluation ===")
test_dataset = DentexDataset(BASE_DIR, ANN_DIR / "instances_test.json", is_train=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=0)

model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()

# --- Run inference on test set ---
test_results = []
test_preds_by_img = {}
with torch.no_grad():
    for imgs, tgts in test_loader:
        imgs = [i.to(device) for i in imgs]
        outs = model(imgs)
        for tgt, out in zip(tgts, outs):
            img_id = int(tgt["image_id"].item())
            preds_list = []
            for b, s, l in zip(out["boxes"].cpu().numpy(), out["scores"].cpu().numpy(), out["labels"].cpu().numpy()):
                test_results.append({"image_id": img_id, "category_id": int(l), "bbox": [float(b[0]), float(b[1]), float(b[2]-b[0]), float(b[3]-b[1])], "score": float(s)})
                preds_list.append({"category_id": int(l), "bbox_xyxy": [float(b[0]), float(b[1]), float(b[2]), float(b[3])], "score": float(s)})
            test_preds_by_img[img_id] = preds_list

# --- COCO mAP (overall) ---
print("\n=== COCO mAP on Held-Out Test Set (50 images) ===")
test_dt = test_dataset.coco.loadRes(test_results)
test_eval = COCOeval(test_dataset.coco, test_dt, "bbox")
test_eval.evaluate()
test_eval.accumulate()
test_eval.summarize()
base_stats = test_eval.stats

# --- Per-Class AP ---
print("\n=== Per-Class AP (from COCOeval precision tensor) ===")
prec = test_eval.eval["precision"]  # shape [T, R, K, A, M]
cat_ids = test_eval.params.catIds
print(f"{'Class':<20} {'AP@50':>8} {'AP@50:95':>10}")
print("-" * 40)
per_class_ap = {}
for k, cid in enumerate(cat_ids):
    a50 = prec[0, :, k, 0, -1]
    a50 = a50[a50 > -1]
    ap50 = float(a50.mean()) if len(a50) > 0 else -1.0
    a_all = prec[:, :, k, 0, -1]
    a_all = a_all[a_all > -1]
    ap_all = float(a_all.mean()) if len(a_all) > 0 else -1.0
    cname = CLASS_NAMES.get(cid, f"class_{cid}")
    per_class_ap[cname] = {"ap50": ap50, "ap50_95": ap_all}
    print(f"  {cname:<18} {ap50:>7.3f} {ap_all:>9.3f}")

# --- Threshold Sweep on Val Set ---
print("\n=== Threshold Sweep (on 70-image val set) ===")
val_results_full = []
val_preds_by_img = {}
with torch.no_grad():
    for imgs, tgts in val_loader:
        imgs = [i.to(device) for i in imgs]
        outs = model(imgs)
        for tgt, out in zip(tgts, outs):
            img_id = int(tgt["image_id"].item())
            pl = []
            for b, s, l in zip(out["boxes"].cpu().numpy(), out["scores"].cpu().numpy(), out["labels"].cpu().numpy()):
                pl.append({"category_id": int(l), "bbox_xyxy": [float(b[0]), float(b[1]), float(b[2]), float(b[3])], "score": float(s)})
            val_preds_by_img[img_id] = pl

def compute_iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0.0

def eval_at_threshold(coco_gt, preds_by_img, thr, iou_thr=0.5):
    tp_total, fp_total, fn_total = 0, 0, 0
    per_cls = {c: {"tp":0,"fp":0,"fn":0} for c in range(1, 5)}
    for img_id in sorted(coco_gt.imgs.keys()):
        anns = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=img_id))
        gt = [(a["bbox"], a["category_id"]) for a in anns]
        gt_xyxy = [([b[0], b[1], b[0]+b[2], b[1]+b[3]], c) for b, c in gt]
        dets = [(p["bbox_xyxy"], p["category_id"], p["score"]) for p in preds_by_img.get(img_id, []) if p["score"] >= thr]
        dets.sort(key=lambda x: -x[2])
        used = set()
        for box, cls, sc in dets:
            best_iou, best_j = 0.0, -1
            for j, (gb, g_cls) in enumerate(gt_xyxy):
                if j in used:
                    continue
                v = compute_iou(box, gb)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= iou_thr and best_j >= 0:
                used.add(best_j)
                if cls == gt_xyxy[best_j][1]:
                    per_cls[cls]["tp"] += 1
                    tp_total += 1
                else:
                    per_cls[cls]["fp"] += 1
                    per_cls[gt_xyxy[best_j][1]]["fn"] += 1
                    fp_total += 1
                    fn_total += 1
            else:
                per_cls[cls]["fp"] += 1
                fp_total += 1
        for j, (gb, g_cls) in enumerate(gt_xyxy):
            if j not in used:
                per_cls[g_cls]["fn"] += 1
                fn_total += 1
    f1s = []
    for c in range(1, 5):
        s = per_cls[c]
        p = s["tp"]/(s["tp"]+s["fp"]) if (s["tp"]+s["fp"]) > 0 else 0.0
        r = s["tp"]/(s["tp"]+s["fn"]) if (s["tp"]+s["fn"]) > 0 else 0.0
        f = 2*p*r/(p+r) if (p+r) > 0 else 0.0
        f1s.append(f)
    return float(np.mean(f1s)), per_cls

best_f1, best_thr = 0.0, 0.3
for thr_candidate in np.arange(0.10, 0.91, 0.05):
    mf1, _ = eval_at_threshold(val_dataset.coco, val_preds_by_img, float(thr_candidate))
    if mf1 > best_f1:
        best_f1, best_thr = mf1, float(thr_candidate)
    print(f"  threshold={thr_candidate:.2f}  macro-F1={mf1:.3f}")
print(f"\nSelected threshold: {best_thr:.2f} (val macro-F1={best_f1:.3f})")
BEST_THRESHOLD = best_thr

# --- Apply to Test Set: Confusion Matrix + P/R/F1 ---
print(f"\n=== Confusion Matrix & P/R/F1 on Test Set (threshold={BEST_THRESHOLD:.2f}) ===")
cm = np.zeros((5, 5), dtype=int)  # rows=true, cols=pred; idx 0=background
for img_id in sorted(test_dataset.coco.imgs.keys()):
    anns = test_dataset.coco.loadAnns(test_dataset.coco.getAnnIds(imgIds=img_id))
    gt = [([a["bbox"][0], a["bbox"][1], a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]], a["category_id"]) for a in anns]
    dets = [(p["bbox_xyxy"], p["category_id"], p["score"]) for p in test_preds_by_img.get(img_id, []) if p["score"] >= BEST_THRESHOLD]
    dets.sort(key=lambda x: -x[2])
    used = set()
    for box, cls, sc in dets:
        best_iou, best_j = 0.0, -1
        for j, (gb, g_cls) in enumerate(gt):
            if j in used:
                continue
            v = compute_iou(box, gb)
            if v > best_iou:
                best_iou, best_j = v, j
        if best_iou >= 0.5 and best_j >= 0:
            used.add(best_j)
            cm[gt[best_j][1], cls] += 1
        else:
            cm[0, cls] += 1
    for j, (gb, g_cls) in enumerate(gt):
        if j not in used:
            cm[g_cls, 0] += 1

labels = ["background", "imp_tooth", "caries", "periapical", "deep_car"]
hdr = "True / Pred"
print(f"\n{hdr:<12}", end="")
for l in labels:
    print(f"{l:>11}", end="")
print()
for i, l in enumerate(labels):
    print(f"{l:<12}", end="")
    for j in range(5):
        print(f"{cm[i,j]:>11d}", end="")
    print()

# Per-class P/R/F1
ch = "Class"
print(f"\n{ch:<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 62)
for cid in range(1, 5):
    cname = CLASS_NAMES[cid]
    tp = cm[cid, cid]
    fp = cm[:, cid].sum() - tp
    fn = cm[cid, :].sum() - tp
    support = cm[cid, :].sum()
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    print(f"  {cname:<18} {p:>9.3f} {r:>9.3f} {f1:>9.3f} {support:>9d}")

# --- Confusion Matrix Heatmap ---
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
cm_display = cm[1:, 1:]  # exclude background row/col for cleaner viz
cls_labels = ["Impacted", "Caries", "Periapical", "Deep Caries"]
im = ax.imshow(cm_display, cmap="Blues", interpolation="nearest")
for r in range(cm_display.shape[0]):
    for c_idx in range(cm_display.shape[1]):
        val = cm_display[r, c_idx]
        color = "white" if val > cm_display.max() * 0.5 else "black"
        ax.text(c_idx, r, str(val), ha="center", va="center", fontsize=14, fontweight="bold", color=color)
ax.set_xticks(range(len(cls_labels)))
ax.set_yticks(range(len(cls_labels)))
ax.set_xticklabels(cls_labels, fontsize=11)
ax.set_yticklabels(cls_labels, fontsize=11)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Ground Truth", fontsize=12)
ax.set_title(f"Confusion Matrix (Test Set, threshold={BEST_THRESHOLD:.2f})", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Sample Detection Visualizations (3 test images) ---
print("\n=== Sample Detection Visualizations (GT=green, Pred=red) ===")
color_map = {1: "#00FF00", 2: "#FF6600", 3: "#FF00FF", 4: "#00CCFF"}
sample_ids = sorted(test_dataset.coco.imgs.keys())[:3]
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax_i, img_id in enumerate(sample_ids):
    info = test_dataset.coco.loadImgs(img_id)[0]
    fname = os.path.basename(info["file_name"])
    pp = list(BASE_DIR.rglob(fname))
    if not pp:
        continue
    img = Image.open(pp[0]).convert("RGB")
    draw = ImageDraw.Draw(img)
    anns = test_dataset.coco.loadAnns(test_dataset.coco.getAnnIds(imgIds=img_id))
    for a in anns:
        x, y, bw, bh = a["bbox"]
        draw.rectangle([x, y, x+bw, y+bh], outline="lime", width=4)
        draw.text((x+2, max(0, y-14)), f"GT:{CLASS_NAMES[a['category_id']]}", fill="lime")
    dets_vis = [(p["bbox_xyxy"], p["category_id"], p["score"]) for p in test_preds_by_img.get(img_id, []) if p["score"] >= BEST_THRESHOLD]
    for box, cls, sc in dets_vis:
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0]+2, max(0, box[1]-14)), f"P:{CLASS_NAMES[cls]} {sc:.2f}", fill="red")
    axes[ax_i].imshow(np.array(img))
    n_gt = len(anns)
    n_det = len(dets_vis)
    axes[ax_i].set_title(f"{fname} (GT={n_gt}, Det={n_det})", fontsize=10)
    axes[ax_i].axis("off")
plt.suptitle("Sample Test Detections: Green=Ground Truth, Red=Predictions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/sample_detections.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 9. Failure Case Analysis — Diverse Cases Across Multiple Images
print("=== Extracting 5 Diverse Failure Cases from Test Set ===")
print("Strategy: max 1 case per image, mix of False Positive / False Negative / Misclassification")
os.makedirs("/kaggle/working/failures", exist_ok=True)

# Collect ALL failure events across the entire test set
all_fps = []   # false positives
all_fns = []   # false negatives
all_mis = []   # misclassifications

for img_id in sorted(test_dataset.coco.imgs.keys()):
    anns = test_dataset.coco.loadAnns(test_dataset.coco.getAnnIds(imgIds=img_id))
    gt = [([a["bbox"][0], a["bbox"][1], a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]], a["category_id"]) for a in anns]
    dets = [(p["bbox_xyxy"], p["category_id"], p["score"]) for p in test_preds_by_img.get(img_id, []) if p["score"] >= BEST_THRESHOLD]
    dets.sort(key=lambda x: -x[2])
    used_gt = set()
    for di, (box, cls, sc) in enumerate(dets):
        best_iou, best_j = 0.0, -1
        for j, (gb, g_cls) in enumerate(gt):
            if j in used_gt:
                continue
            v = compute_iou(box, gb)
            if v > best_iou:
                best_iou, best_j = v, j
        if best_iou >= 0.5 and best_j >= 0:
            used_gt.add(best_j)
            if cls != gt[best_j][1]:
                all_mis.append({"img_id": img_id, "type": "Misclassification", "score": sc,
                    "gt_label": gt[best_j][1], "pred_label": cls,
                    "detail": f"GT={CLASS_NAMES[gt[best_j][1]]}, Pred={CLASS_NAMES[cls]} (score={sc:.2f})"})
        else:
            all_fps.append({"img_id": img_id, "type": "False Positive", "score": sc,
                "pred_label": cls,
                "detail": f"Pred={CLASS_NAMES[cls]} (score={sc:.2f}), no matching GT (IoU={best_iou:.2f})"})
    for j, (gb, g_cls) in enumerate(gt):
        if j not in used_gt:
            all_fns.append({"img_id": img_id, "type": "False Negative", "score": 0.0, "gt_label": g_cls,
                "detail": f"Missed GT={CLASS_NAMES[g_cls]} — no detection with IoU>=0.5"})

print(f"\nTotal failure events: {len(all_fps)} FP, {len(all_fns)} FN, {len(all_mis)} Misclass")
print(f"Unique images with FP: {len(set(f['img_id'] for f in all_fps))}")
print(f"Unique images with FN: {len(set(f['img_id'] for f in all_fns))}")

# Select 5 diverse cases: pick highest-confidence FP, worst FN, any misclass — one per image
failures = []
used_images = set()

# 1. Pick top FP (highest confidence false alarm)
all_fps.sort(key=lambda x: -x["score"])
for fp in all_fps:
    if fp["img_id"] not in used_images:
        failures.append(fp)
        used_images.add(fp["img_id"])
        break

# 2. Pick a FN from a different image
for fn in all_fns:
    if fn["img_id"] not in used_images:
        failures.append(fn)
        used_images.add(fn["img_id"])
        break

# 3. Pick a misclassification if available
for mc in all_mis:
    if mc["img_id"] not in used_images:
        failures.append(mc)
        used_images.add(mc["img_id"])
        break

# 4-5. Fill remaining with alternating FP/FN from unused images
for pool in [all_fns, all_fps, all_fns, all_fps]:
    if len(failures) >= 5:
        break
    for item in pool:
        if item["img_id"] not in used_images:
            failures.append(item)
            used_images.add(item["img_id"])
            break

# Draw annotated images for each failure case
for fi, fail in enumerate(failures[:5]):
    img_id = fail["img_id"]
    info = test_dataset.coco.loadImgs(img_id)[0]
    fname = os.path.basename(info["file_name"])
    pp = list(BASE_DIR.rglob(fname))
    if not pp:
        continue
    img = Image.open(pp[0]).convert("RGB")
    draw = ImageDraw.Draw(img)
    anns = test_dataset.coco.loadAnns(test_dataset.coco.getAnnIds(imgIds=img_id))
    for a in anns:
        x, y, bw, bh = a["bbox"]
        draw.rectangle([x, y, x+bw, y+bh], outline="lime", width=4)
        draw.text((x+2, max(0, y-14)), f"GT:{CLASS_NAMES[a['category_id']]}", fill="lime")
    dets_draw = [(p["bbox_xyxy"], p["category_id"], p["score"]) for p in test_preds_by_img.get(img_id, []) if p["score"] >= BEST_THRESHOLD]
    for box, cls, sc in dets_draw:
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0]+2, max(0, box[1]-14)), f"P:{CLASS_NAMES[cls]} {sc:.2f}", fill="red")
    out_path = f"/kaggle/working/failures/case_{fi+1}.png"
    img.save(out_path)
    print(f"\nCase {fi+1}: {fail['type']}")
    print(f"  Image: {fname}")
    print(f"  {fail['detail']}")

# Plot all failure cases side by side
n_cases = min(5, len(failures))
fig, axes = plt.subplots(1, n_cases, figsize=(6*n_cases, 6))
if n_cases == 1:
    axes = [axes]
type_colors = {"False Positive": "#FF4444", "False Negative": "#FF8C00", "Misclassification": "#FF00FF"}
for i, ax in enumerate(axes):
    fp = f"/kaggle/working/failures/case_{i+1}.png"
    if os.path.exists(fp):
        ax.imshow(np.array(Image.open(fp)))
        ftype = failures[i]["type"]
        tc = type_colors.get(ftype, "black")
        ax.set_title(f"Case {i+1}: {ftype}", fontsize=11, fontweight="bold", color=tc)
    ax.axis("off")
plt.suptitle("Failure Case Analysis — Green=GT, Red=Predictions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/failure_cases.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Failure Analysis Summary (data-driven) ===")

# Compute FP breakdown by predicted class
fp_class_counts = {}
for f in all_fps:
    cname = CLASS_NAMES.get(f['pred_label'], f"class_{f['pred_label']}")
    fp_class_counts[cname] = fp_class_counts.get(cname, 0) + 1
print(f"False Positives: {len(all_fps)} total across {len(set(f['img_id'] for f in all_fps))} images")
for cname, cnt in sorted(fp_class_counts.items(), key=lambda x: -x[1]):
    print(f"  FP predicted as {cname}: {cnt}")

# Compute FN breakdown by missed GT class
fn_class_counts = {}
for f in all_fns:
    cname = CLASS_NAMES.get(f['gt_label'], f"class_{f['gt_label']}")
    fn_class_counts[cname] = fn_class_counts.get(cname, 0) + 1
print(f"\nFalse Negatives: {len(all_fns)} total across {len(set(f['img_id'] for f in all_fns))} images")
for cname, cnt in sorted(fn_class_counts.items(), key=lambda x: -x[1]):
    print(f"  Missed GT {cname}: {cnt}")

# Misclassification breakdown
if all_mis:
    mis_pairs = {}
    for f in all_mis:
        gt_name = CLASS_NAMES.get(f['gt_label'], '?')
        pr_name = CLASS_NAMES.get(f['pred_label'], '?')
        pair = f"{gt_name} -> {pr_name}"
        mis_pairs[pair] = mis_pairs.get(pair, 0) + 1
    print(f"\nMisclassifications: {len(all_mis)} total")
    for pair, cnt in sorted(mis_pairs.items(), key=lambda x: -x[1]):
        print(f"  {pair}: {cnt}")


In [ ]:
# 10. Improvement Experiment: Trained Variant (960x1600) — same split, seed, epochs
import gc  # re-import in case prior cells shadowed it
print("=== Training Variant Model (min_size=960, max_size=1600) ===")

# Clean up baseline model from GPU
try:
    del optimizer, lr_scheduler, scaler
except NameError:
    pass
del model  # free baseline model GPU memory before loading variant
torch.cuda.empty_cache()
gc.collect()

# Save baseline stats for comparison
baseline_test_stats = base_stats.copy()

# New model at higher resolution — batch_size=1 to avoid OOM with ResNet-50 at 960x1600
model_v = get_model(min_size=960, max_size=1600).to(device)
opt_v = torch.optim.SGD([p for p in model_v.parameters() if p.requires_grad], lr=LR, momentum=0.9, weight_decay=0.0005)
sched_v = torch.optim.lr_scheduler.CosineAnnealingLR(opt_v, T_max=EPOCHS, eta_min=LR * 0.01)
scaler_v = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

# Re-seed for reproducibility
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
g_v = torch.Generator()
g_v.manual_seed(SEED)
# batch_size=1 for higher resolution to fit T4 GPU memory (16GB)
train_loader_v = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn, generator=g_v, worker_init_fn=worker_init_fn, num_workers=2)

print(f"Variant: batch_size=1, CosineAnnealingLR, AMP={device.type=='cuda'}")

best_map_v = 0.0
for epoch in range(1, EPOCHS + 1):
    model_v.train()
    loss_sum = 0.0
    for imgs, tgts in train_loader_v:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        opt_v.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            ld = model_v(imgs, tgts)
            loss = sum(v for v in ld.values())
        scaler_v.scale(loss).backward()
        scaler_v.step(opt_v)
        scaler_v.update()
        loss_sum += loss.item()
    sched_v.step()
    ml = loss_sum / len(train_loader_v)
    cur_map_v, st_v = run_val_eval(model_v, val_loader, val_dataset)
    ar_v = float(st_v[8]) if st_v is not None else 0.0
    print(f"  Variant Epoch [{epoch}/{EPOCHS}] Loss: {ml:.4f} | Val mAP@50: {cur_map_v:.3f} | AR@100: {ar_v:.3f}")
    if cur_map_v >= best_map_v:
        best_map_v = cur_map_v
        torch.save(model_v.state_dict(), "/kaggle/working/best_variant.pth")

# Evaluate variant on held-out test set
model_v.load_state_dict(torch.load("/kaggle/working/best_variant.pth"))
model_v.eval()
var_test_results = []
with torch.no_grad():
    for imgs, tgts in test_loader:
        imgs = [i.to(device) for i in imgs]
        outs = model_v(imgs)
        for tgt, out in zip(tgts, outs):
            img_id = int(tgt["image_id"].item())
            for b, s, l in zip(out["boxes"].cpu().numpy(), out["scores"].cpu().numpy(), out["labels"].cpu().numpy()):
                var_test_results.append({"image_id": img_id, "category_id": int(l), "bbox": [float(b[0]), float(b[1]), float(b[2]-b[0]), float(b[3]-b[1])], "score": float(s)})

var_dt = test_dataset.coco.loadRes(var_test_results)
var_eval = COCOeval(test_dataset.coco, var_dt, "bbox")
var_eval.evaluate()
var_eval.accumulate()
var_eval.summarize()
var_stats = var_eval.stats

# Per-class AP for variant
var_prec = var_eval.eval["precision"]
var_cat_ids = var_eval.params.catIds

# Save variant config for reproducibility
var_config = config.copy()
var_config["model"] = dict(config["model"])
var_config["model"]["min_size"] = 960
var_config["model"]["max_size"] = 1600
var_config["training"] = dict(config["training"])
var_config["training"]["batch_size"] = 1
var_config["training"]["note"] = "batch_size reduced from 2 to 1 to avoid OOM at higher resolution"
with open("/kaggle/working/config_variant.yaml", "w") as f:
    yaml.dump(var_config, f, default_flow_style=False)
print("Saved /kaggle/working/config_variant.yaml")

# --- Honest Comparison Table ---
print(f"\n{'='*70}")
print(f"  Improvement Experiment: Baseline (800x1333) vs Trained Variant (960x1600)")
print(f"  Both trained for {EPOCHS} epochs on same 635-image train set, seed={SEED}")
print(f"  NOTE: Experiment changes TWO variables simultaneously:")
print(f"    1. Input resolution: 800x1333 -> 960x1600")
print(f"    2. Batch size: 2 -> 1 (to avoid GPU OOM at higher resolution)")
print(f"    The batch size change affects optimizer dynamics (2x steps at same LR).")
print(f"    A controlled experiment would require gradient accumulation.")
print(f"  Test set used for both models below (not used for threshold/checkpoint selection).")
print(f"{'='*70}")
print(f"{'Metric':<25} {'Baseline':>12} {'Variant':>12} {'Delta':>12}")
print("-" * 65)
metric_names = [("mAP@50", 1), ("mAP@50:95", 0), ("mAP@75", 2), ("AR@1", 6), ("AR@10", 7), ("AR@100", 8)]
for name, idx in metric_names:
    bv = float(baseline_test_stats[idx])
    vv = float(var_stats[idx])
    d = vv - bv
    print(f"  {name:<23} {bv:>11.3f} {vv:>11.3f} {d:>+11.3f}")

# Per-class AP comparison
print(f"\n{'Per-Class AP@50':<25} {'Baseline':>12} {'Variant':>12} {'Delta':>12}")
print("-" * 65)
base_prec = test_eval.eval["precision"]
for k, cid in enumerate(cat_ids):
    cname = CLASS_NAMES.get(cid, f"class_{cid}")
    b50 = base_prec[0,:,k,0,-1]; b50 = float(b50[b50>-1].mean()) if (b50>-1).any() else 0.0
    v50 = var_prec[0,:,k,0,-1]; v50 = float(v50[v50>-1].mean()) if (v50>-1).any() else 0.0
    print(f"  {cname:<23} {b50:>11.3f} {v50:>11.3f} {v50-b50:>+11.3f}")

print(f"\nVariant checkpoint: {os.path.getsize('/kaggle/working/best_variant.pth'):,} bytes")

# --- Final Model Declaration ---
print(f"\n=== Final Model Declaration ===")
print(f"Selected model: Baseline (800x1333, batch_size=2)")
print(f"Rationale: Higher mAP@50:95 ({baseline_test_stats[0]:.3f} vs {var_stats[0]:.3f})")
print(f"           and better mAP@75 ({baseline_test_stats[2]:.3f} vs {var_stats[2]:.3f})")
print(f"           indicating more precise box localisation.")
print(f"Checkpoint: /kaggle/working/best_model.pth")
print(f"Operating threshold: {BEST_THRESHOLD:.2f} (selected by macro-F1 sweep on val set)")


In [ ]:
# 11. CPU Latency Benchmark (Real OPG, End-to-End)
print("=== CPU Inference Latency (Real Image, Operating Threshold) ===")

# Build model with operating threshold for realistic latency
cpu_model = get_model(min_size=800, max_size=1333, score_thresh=BEST_THRESHOLD).to("cpu")
cpu_model.load_state_dict(torch.load("/kaggle/working/best_model.pth", map_location="cpu"))
cpu_model.eval()

# Find a real test image
test_img_info = test_dataset.coco.loadImgs(test_dataset.ids[0])[0]
test_fname = os.path.basename(test_img_info["file_name"])
test_img_paths = list(BASE_DIR.rglob(test_fname))
test_img_path = test_img_paths[0] if test_img_paths else None

if test_img_path:
    sample_img = Image.open(test_img_path).convert("RGB")
    w, h = sample_img.size
    print(f"Test image: {test_fname} ({w}x{h})")
else:
    print("No test image found, using synthetic input")
    w, h = 2943, 1435

print(f"Operating threshold: {BEST_THRESHOLD:.2f}")
print(f"torch.get_num_threads(): {torch.get_num_threads()}")
!cat /proc/cpuinfo | grep "model name" | head -1

# Warmup
for _ in range(5):
    if test_img_path:
        img = Image.open(test_img_path).convert("RGB")
        tensor = transforms.functional.to_tensor(img)
    else:
        tensor = torch.randn(3, h, w)
    with torch.no_grad():
        _ = cpu_model([tensor])

# Timed runs (end-to-end: disk read + decode + to_tensor + forward + NMS)
times = []
for _ in range(30):
    t0 = time.perf_counter()
    if test_img_path:
        img = Image.open(test_img_path).convert("RGB")
        tensor = transforms.functional.to_tensor(img)
    else:
        tensor = torch.randn(3, h, w)
    with torch.no_grad():
        outputs = cpu_model([tensor])
    times.append((time.perf_counter() - t0) * 1000.0)

times_arr = np.array(times)
print(f"\nLatency over {len(times)} runs (including image load + decode):")
print(f"  Mean:   {times_arr.mean():.1f} ms")
print(f"  Median: {np.median(times_arr):.1f} ms")
print(f"  P50:    {np.percentile(times_arr, 50):.1f} ms")
print(f"  P95:    {np.percentile(times_arr, 95):.1f} ms")
print(f"  Min:    {times_arr.min():.1f} ms")
print(f"  Max:    {times_arr.max():.1f} ms")
print(f"  Throughput: {1000.0 / times_arr.mean():.2f} images/sec")

try:
    import resource
    peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    print(f"  Peak RSS: {peak_kb / 1024:.0f} MB")
except Exception:
    pass
